# common/preprocessing — Phân nhóm cột và pipeline tiền xử lý

Notebook chứa hàm dùng chung, không tự train và không tự ghi kết quả. Các notebook chính nạp tự động bằng `%run`. Khi sửa hàm, khởi động lại kernel rồi Run All notebook chính.


## Nạp các hàm phụ thuộc
Cell này cũng cho phép mở và Run All riêng notebook dùng chung.

In [ ]:
from pathlib import Path
import os
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'common/data_io.ipynb').is_file()), None)
if root is None:
    raise FileNotFoundError('Mở notebook trong thư mục project đầy đủ.')
os.chdir(root)
get_ipython().run_line_magic('run', 'common/data_io.ipynb')


## Thư viện và hằng số

In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler


## `feature_groups`
Phân biệt cột số và các cột mã hóa dạng phân loại (gồm cả nhị phân).

In [ ]:
def feature_groups(df):
    cats = [c for c in df.columns if c != TARGET and (c in CATEGORICAL or c in BINARY or not pd.api.types.is_numeric_dtype(df[c]))]
    nums = [c for c in df.columns if c not in cats and c != TARGET]
    return nums, cats


## `make_preprocessor`
Điền thiếu, chuẩn hóa và one-hot trong pipeline để fit riêng trên từng fold.

In [ ]:
def make_preprocessor(df):
    nums, cats = feature_groups(df)
    # sparse=False works on scikit-learn <1.2, sparse_output on newer versions.
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    transformers = []
    if nums:
        transformers.append(('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), nums))
    if cats:
        transformers.append(('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', encoder)]), cats))
    return ColumnTransformer(transformers)
